# 캐글 필사 — Spaceship Titanic (분류)

ESAA YB · 2026-11-09

**원본**: [Spaceship Titanic | EDA + Model - 100%](https://www.kaggle.com/code/lightonkalumba/spaceship-titanic-eda-model-100) by Lighton N. kalumba
**대회**: [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic) — 승객이 다른 차원으로 이동(`Transported`)했는지 맞히는 이진 분류
**원본 Public Score**: 0.69628

원본 코드를 그대로 따라 치면서, 각 단계가 무엇을 하는지와 **고칠 부분**을 함께 적었다.
제목에 100%가 붙어 있지만 리더보드 점수는 0.69628이다. 왜 그런 차이가 나는지가
이 노트북에서 가장 배울 점이라 6절에서 따로 짚는다.

## 0. 데이터 준비 (코랩용)

원본은 캐글에서 돌아가므로 `/kaggle/input/spaceship-titanic/`을 읽는다.
코랩에서는 그 경로가 없으니, 대회 페이지에서 `train.csv`와 `test.csv`를 받아 올린다.

In [ ]:
import os

DATA_DIR = '/kaggle/input/spaceship-titanic'   # 캐글이면 그대로 사용

if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'
    if not os.path.exists('train.csv'):
        # 캐글 API 사용: kaggle.json을 올린 뒤 실행
        from google.colab import files
        print('train.csv, test.csv를 올려주세요 (캐글 대회 페이지 > Data)')
        files.upload()

print('데이터 경로:', DATA_DIR)
print([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])

## 1. 라이브러리 불러오기

In [ ]:
#Import basic Libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import SimpleImputer

## 2. 데이터 불러오기

In [ ]:
#Load Data
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

## 3. 기초 탐색

In [ ]:
#Load the test Data
test.head()

In [ ]:
#Load the train Data
train.head()

In [ ]:
train.info()

변수는 14개다.

| 변수 | 뜻 |
|---|---|
| `PassengerId` | `gggg_pp` 형식. 앞이 일행 번호, 뒤가 일행 내 순번 |
| `HomePlanet` | 출발 행성 |
| `CryoSleep` | 냉동수면 여부 |
| `Cabin` | `deck/num/side` 형식의 객실 번호 |
| `Destination` | 목적지 |
| `Age` | 나이 |
| `VIP` | VIP 여부 |
| `RoomService` ~ `VRDeck` | 시설별 지출액 (5개) |
| `Name` | 이름 |
| `Transported` | **타깃** — 다른 차원으로 이동했는지 |

`PassengerId`와 `Cabin`은 **여러 정보가 합쳐진 문자열**이다. 쪼개면 쓸 수 있는 변수가 되는데,
원본은 그대로 둔다. 이 부분은 6절에서 다시 본다.

In [ ]:
#Percentage of missing values in the dataset
missing_percentage = (train.isnull().sum() / len(train)) * 100
print(missing_percentage)

In [ ]:
#Percentage of missing values in the dataset
missing_percentage = (test.isnull().sum() / len(test)) * 100
print(missing_percentage)

결측값이 거의 모든 열에 2% 안팎으로 퍼져 있다. 한 열에 몰려 있지 않고
여러 열에 조금씩 있는 형태라, 결측 행을 지우면 데이터가 꽤 줄어든다.
그래서 지우지 않고 채우는 쪽을 택한다.

## 4. 탐색적 데이터 분석

In [ ]:
true_count = train['Transported'].sum()
false_count = len(train) - true_count

# Data for the pie chart
labels = ['True', 'False']
sizes = [true_count, false_count]
colors = ['skyblue', 'lightcoral']
explode = (0.1, 0)

# Create the pie chart
plt.pie(sizes, explode=explode, labels=labels, colors=colors,
autopct='%1.1f%%', shadow=True, startangle=140)

plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.title("Transported Column Distribution", size = 15)
plt.show()

타깃이 거의 50 대 50이다. **불균형이 없는 문제**라 정확도를 그대로 지표로 써도 된다.
전부 한쪽으로 찍는 기준선이 약 0.50이므로, 모델은 그보다 얼마나 높은지로 평가한다.

미니프로젝트 와인 데이터는 6점이 44%라 기준선이 0.4395였는데, 여기는 그런 고민이 없다.

In [ ]:
age_data = train['Age']

plt.figure(figsize=(10, 6))
plt.hist(age_data, bins=20, edgecolor='black', color='#6A0572', alpha=0.7, label='Age Distribution')

plt.title('Age Distribution', fontsize=18, fontweight='bold', color='black')
plt.xlabel('Age', fontsize=14, color='black')
plt.ylabel('Frequency', fontsize=14, color='black')

plt.xticks(fontsize=12, color='black')
plt.yticks(fontsize=12, color='black')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.gca().set_facecolor('black')

# Remove unnecessary spines and set the layout
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['left'].set_linewidth(0.5)
plt.gca().spines['bottom'].set_linewidth(0.5)

# Add legend
plt.legend()

plt.tight_layout()

plt.show()

20~30대에 몰려 있고 0세 근처에 봉우리가 하나 더 있다.
영유아는 지출이 0일 수밖에 없어서 지출 변수들과 묶어 보면 패턴이 나올 수 있다.

In [ ]:
sex_counts = train['VIP'].value_counts()
colors = ['#3498db', '#e74c3c']

# Create the pie chart
plt.figure(figsize=(6, 6))
plt.pie(sex_counts, labels=sex_counts.index, colors=colors, autopct='%1.1f%%', startangle=90, shadow=True, wedgeprops={'linewidth': 2, 'edgecolor': 'white'})

# Add a title and legend
plt.title("Distribution of VIP Status in the Dataset", fontsize=16)
plt.legend(labels=sex_counts.index, loc='upper right')

# Equal aspect ratio ensures that pie is drawn as a circle.
plt.axis('equal')

# Show the pie chart
plt.tight_layout()
plt.show()

VIP는 2% 남짓이다. 이렇게 한쪽으로 치우친 변수는 영향력이 작을 가능성이 크다.
17번 셀의 변수 중요도에서 실제로 확인된다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

food_court = train['FoodCourt']
home_planet = train['HomePlanet']

label_encoder = LabelEncoder()
home_planet_numerical = label_encoder.fit_transform(home_planet)

# Create a scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(food_court, home_planet_numerical, alpha=0.5)

plt.xlabel('Food Court')
plt.ylabel('Home Planet (Numerical)')
plt.title('Scatter Plot: Food Court vs. Home Planet')

# Show the plot
plt.show()

세로축이 행성을 0, 1, 2로 바꾼 값이라 **산점도로 그릴 대상이 아니다.**
범주형이라 점들이 세 줄에 걸쳐 찍힐 뿐이고, 세로 방향의 거리에 의미가 없다.

같은 내용을 보려면 행성별 `FoodCourt` 분포를 박스플롯으로 그리는 편이 낫다.
다만 그래프 모양에서 **행성마다 지출 범위가 다르다**는 것 정도는 읽을 수 있다.

## 5. 데이터 전처리

In [ ]:
categorical_columns = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']

# Create a LabelEncoder for each categorical column
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    label_encoders[col] = le

# Sample data with encoded categorical columns
X_categorical_encoded = train[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']]
y = train['Transported']

# Concatenate the encoded categorical columns with other numerical columns
X_encoded = pd.concat([X_categorical_encoded, train[['Age']]], axis=1)

# Create an imputer with median strategy
imputer = SimpleImputer(strategy='median')

# Fit the imputer on the data and transform to fill missing values
X_imputed = imputer.fit_transform(X_encoded)

# Convert the imputed array back to a DataFrame with column names
X_imputed_df = pd.DataFrame(X_imputed, columns=X_encoded.columns)

# Assign the imputed DataFrame back to X
X = X_imputed_df

# Create a LabelEncoder for the target variable 'Transported'
label_encoder_y = LabelEncoder()
y_encoded = label_encoder_y.fit_transform(y)

# Assign the encoded target variable back to y
y = y_encoded

이 셀이 이 노트북에서 가장 문제가 많은 부분이다. 세 가지를 짚어둔다.

**① 연속형 변수에 `LabelEncoder`를 씌웠다.**
`categorical_columns`에 `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`이 들어가 있다.
이건 **지출 금액**이라 연속형이다. `LabelEncoder`는 값을 정렬해서 0부터 번호를 매기므로,
지출액이 등수로 바뀐다. 0원과 10원의 간격이 10원과 5,000원의 간격과 같아진다.

**② `Name`을 변수로 썼다.**
이름은 승객마다 거의 다르다. 번호를 매기면 사실상 ID가 되고, 모델은 그 번호를 외운다.
학습 데이터에서는 잘 맞고 새 데이터에서는 안 맞는 전형적인 과적합 변수다.

**③ `Cabin`을 통째로 번호 매겼다.**
`Cabin`은 `B/0/P`처럼 **갑판 / 번호 / 좌우**가 합쳐진 값이다. 통째로 인코딩하면
6,000종이 넘는 값이 생기고 갑판 정보가 사라진다. `/`로 쪼개서 세 변수로 만드는 것이 보통이다.

`SimpleImputer(strategy='median')`로 결측값을 중앙값으로 채우는 것 자체는 무난하다.
다만 `LabelEncoder`를 결측값이 있는 상태로 돌렸기 때문에, **결측값도 하나의 범주로
번호를 받은 뒤** 중앙값 대체가 적용된다. 순서가 뒤바뀌어 있다.

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k='all')
X_new = selector.fit_transform(X, y)

# Print the selected features and their scores

selected_features = X.columns[selector.get_support()]
scores = selector.scores_[selector.get_support()]
feature_scores = list(zip(selected_features, scores))
print("Selected Features and their Scores:")
for feature, score in feature_scores:
    print(f"{feature}: {score:.2f}")

`SelectKBest(f_classif)`는 각 변수가 타깃을 혼자서 얼마나 구분하는지 F값으로 재준다.
`k='all'`이라 거르지는 않고 점수만 출력한다.

점수가 높게 나오는 변수와 낮게 나오는 변수를 보되, **낮다고 바로 빼면 안 된다.**
미니프로젝트에서 상관이 약한 변수 7개를 뺐더니 정확도가 0.6687에서 0.6427로 떨어졌다.
혼자서는 설명력이 없어도 다른 변수와 함께 쓰이면 기여할 수 있기 때문이다.

## 6. 모델링

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.001, random_state=15)

**`test_size = 0.001`이 이 노트북의 핵심 문제다.**

전체 8,693행의 0.1%면 검증 데이터가 **9행**이다. 9개를 맞히는 정확도로
모델 성능을 말할 수는 없다. 9개 중 9개를 맞히면 100%, 8개를 맞히면 88.9%가 된다.
제목의 "100%"는 여기서 나온 숫자다.

**실제 리더보드 점수는 0.69628이다.** 30%p 넘게 차이가 난다.

미니프로젝트에서 확인한 것과 같은 문제다. 홀드아웃 20%로 재도 `random_state`만 바꾸면
점수가 4.2%p씩 움직였고, 그래서 교차검증을 쓰기로 했다. 여기는 0.1%이니 훨씬 심하다.

```python
# 이렇게 바꾸는 것이 맞다
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv)
print(scores.mean(), scores.std())
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model

In [ ]:
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

여기서 나오는 숫자가 제목의 "100%"다. **9개짜리 검증 결과**라는 점을 기억해야 한다.

### 교차검증으로 다시 재보기

원본에는 없지만, 실제 성능이 얼마인지 확인하려고 추가했다.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=cv)

print('교차검증 정확도: %.4f (±%.4f)' % (scores.mean(), scores.std()))
print('폴드별:', scores.round(4).tolist())
print()
print('홀드아웃 9개로 잰 점수: %.4f' % model.score(X_test, y_test))
print('실제 리더보드 점수   : 0.69628')

## 7. 제출 파일 만들기

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Load the test dataset
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# Apply the same LabelEncoder transformations to categorical columns
categorical_columns = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']

for col in categorical_columns:
    if col in label_encoders:
        le = label_encoders[col]
        # Transform test data and handle unknown categories by assigning a specific label (-1 in this case)
        test[col] = test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

# Select the same columns as you did for training data
X_test_categorical_encoded = test[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']]

# Concatenate the encoded categorical columns with other numerical columns
X_test_encoded = pd.concat([X_test_categorical_encoded, test[['Age']]], axis=1)

# Use the same imputer as in the training data
imputer = SimpleImputer(strategy='median')
X_test_imputed = imputer.fit_transform(X_test_encoded)

# Convert the imputed array back to a DataFrame with column names
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=X_test_encoded.columns)

# Now, X_test_imputed_df contains the preprocessed test data

# Make predictions on the test data using your trained model
predictions = model.predict(X_test_imputed_df)  # Replace 'model' with your trained model

# Create a submission DataFrame
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Transported': predictions})

주석에는 "Use the same imputer as in the training data"라고 적혀 있는데,
코드는 **`SimpleImputer()`를 새로 만들어 test에 다시 `fit_transform`** 한다.

그러면 train의 중앙값이 아니라 **test의 중앙값**으로 채워진다. 두 데이터의 중앙값이
다르면 같은 결측값이 서로 다른 값으로 바뀐다. 학습할 때와 예측할 때 기준이 달라지는 것이다.

```python
# 이렇게 해야 한다 — train에서 학습한 기준을 test에 적용
X_test_imputed = imputer.transform(X_test_encoded)   # fit_transform이 아니라 transform
```

미니프로젝트 공통 규칙으로 정한 "데이터에서 기준을 학습하는 처리는 폴드 안에서"가
바로 이 문제를 막기 위한 것이다.

In [ ]:
target_map = {0: 'False', 1: 'True'}

final_df = pd.DataFrame(submission)
final_df['Transported'] = pd.Series(predictions).map(target_map)
final_df.to_csv('submission.csv', index=False)

In [ ]:
final_df

`Transported`를 문자열 `'True'`/`'False'`로 바꿔 저장한다.
이 대회는 불리언을 받으므로 문자열로 내도 채점이 된다.

`index=False`를 빠뜨리면 열이 하나 더 생겨 형식 오류가 난다.

## 느낀 점

**숫자 하나만 보고 성능을 말하면 안 된다.** 제목은 100%인데 리더보드는 0.69628이다.
`test_size=0.001`로 9개만 떼어 재서 생긴 차이다. 미니프로젝트에서 홀드아웃이
`random_state`만 바꿔도 4.2%p 흔들리는 걸 확인하고 교차검증으로 통일했는데,
그 결정이 왜 필요한지 이 노트북이 극단적인 예로 보여준다.

**전처리는 변수의 성격을 보고 골라야 한다.** `LabelEncoder`를 범주형이라는 이름만 보고
지출 금액 5개에도 적용했다. 금액이 등수로 바뀌면 0원과 10원의 차이가 10원과 5,000원의
차이와 같아진다. 함수 이름이 아니라 **그 변수가 무엇을 재는 값인지**를 봐야 한다.

**이름 같은 식별자는 변수가 아니다.** `Name`은 승객마다 달라서 번호를 매기면 ID가 된다.
학습 데이터는 외워서 맞히고 새 데이터에서는 못 맞힌다. 연습문제2에서 `index`를 지운 것과
같은 이유인데, 여기서는 오히려 추가했다.

**합쳐진 값은 쪼개야 쓸 수 있다.** `Cabin`은 `갑판/번호/좌우`가 합쳐진 문자열이고
`PassengerId`는 `일행번호_순번`이다. 통째로 인코딩하면 수천 개의 무의미한 번호가 되지만,
쪼개면 갑판과 일행 정보를 얻는다. 상위 노트북들이 점수 차이를 내는 지점이 대체로 여기다.

**주석과 코드가 다를 수 있다.** 제출 부분 주석은 "같은 imputer를 쓴다"인데 코드는
새로 만들어 `fit_transform` 한다. 남의 코드를 읽을 때는 주석이 아니라 코드를 봐야 한다.

종합하면, 이 노트북은 **흐름을 익히기에는 좋지만 그대로 따라 하면 안 되는** 예시다.
전처리 → 모델 → 제출의 뼈대는 분명한데, 각 단계의 선택이 변수의 성격과 맞지 않는다.
필사하면서 "왜 이렇게 했을까"를 묻는 것이 코드를 그대로 옮기는 것보다 남는 게 많았다.